# Step 10: Relation-Chain Artifact Generation

这个 notebook 只为 `hp_relation_chain_bridge_set_01` 生成两类 memory artifact：

- `episodic_trace.md`
- `cross_episode_consolidation.md`

它和 [04_artifact_generation.ipynb](./04_artifact_generation.ipynb) 的区别是：

- `04` 固定读 Round 1 frozen archive
- `10` 读 working `source_sets.csv` 和 Batch 2 expansion results
- `10` 默认只处理 `hp_relation_chain_bridge_set_01`


In [1]:
import csv
import gc
import json
import os
from pathlib import Path

try:
    import torch
except Exception:
    torch = None

try:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
except Exception:
    AutoModelForCausalLM = None
    AutoTokenizer = None
    BitsAndBytesConfig = None


## 1. 配置路径与生成参数


In [2]:
PROJECT_ROOT_OVERRIDE = ''

MODEL_ID = 'Qwen/Qwen3.5-9B'
FALLBACK_MODEL_ID = 'Qwen/Qwen3.5-4B'
HF_TOKEN = os.environ.get('HF_TOKEN', '')
RUN_GENERATION = True
USE_4BIT = False
ENABLE_THINKING = False
MAX_NEW_TOKENS = 1400
DO_SAMPLE = False
TARGET_SOURCE_SET_IDS = ['hp_relation_chain_bridge_set_01']


def candidate_roots():
    candidates = []
    if PROJECT_ROOT_OVERRIDE.strip():
        candidates.append(Path(PROJECT_ROOT_OVERRIDE).expanduser())

    env_root = os.environ.get('SELECT_TRANSFER_ROOT', '').strip()
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd.parent,
        cwd / '2026_SelectTransfer',
        Path('/content/2026_SelectTransfer'),
        Path('/workspace/2026_SelectTransfer'),
        Path('/root/2026_SelectTransfer'),
        Path('/kaggle/working/2026_SelectTransfer'),
    ])

    seen = set()
    ordered = []
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        ordered.append(candidate)
    return ordered


def detect_project_root():
    checked = []
    for candidate in candidate_roots():
        checked.append(str(candidate))
        if candidate.exists() and (candidate / 'pilot').exists() and (candidate / 'results').exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate project root. Set PROJECT_ROOT_OVERRIDE or SELECT_TRANSFER_ROOT to the uploaded 2026_SelectTransfer directory. Checked: ' + ' | '.join(checked)
    )


PROJECT_ROOT = detect_project_root()
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_SETS_PATH = PROJECT_ROOT / 'pilot' / 'source_sets.csv'
BASE_TAXONOMY_PATH = PROJECT_ROOT / 'pilot' / 'taxonomy.csv'
BATCH2_ANNOTATION_PATH = PROJECT_ROOT / 'results' / '09_relation_chain_bridge_expansion_batch2' / 'candidate_batch2_for_subtype_annotation.csv'

SAMPLED_JSON_PATH = PROJECT_ROOT / 'results' / '01_sampling' / 'sampled_20_full.json'
EXPANDED_JSON_PATH = PROJECT_ROOT / 'results' / '02_hotpotqa_comparison_expansion' / 'candidate_batch_filtered_full.json'
BATCH2_FULL_JSON_PATH = PROJECT_ROOT / 'results' / '09_relation_chain_bridge_expansion_batch2' / 'candidate_batch2_full.json'

MANIFEST_PATH = ARTIFACTS_DIR / 'relation_chain_artifact_generation_manifest.csv'

REQUIRED_INPUTS = {
    'working source sets': SOURCE_SETS_PATH,
    'working taxonomy': BASE_TAXONOMY_PATH,
    'Batch 2 subtype annotation': BATCH2_ANNOTATION_PATH,
    'Round 1 sampled payload json': SAMPLED_JSON_PATH,
    'comparison expansion payload json': EXPANDED_JSON_PATH,
    'relation-chain Batch 2 payload json': BATCH2_FULL_JSON_PATH,
}

missing_inputs = {name: str(path) for name, path in REQUIRED_INPUTS.items() if not path.exists()}
if missing_inputs:
    missing_lines = [f'- {name}: {path}' for name, path in missing_inputs.items()]
    raise FileNotFoundError(
        'Detected project root but some required inputs are missing:\n' + '\n'.join(missing_lines)
    )

print('PROJECT_ROOT =', PROJECT_ROOT)
print('MODEL_ID =', MODEL_ID)
print('TARGET_SOURCE_SET_IDS =', TARGET_SOURCE_SET_IDS)
print('required inputs checked =', len(REQUIRED_INPUTS))


PROJECT_ROOT = /root/2026_SelectTransfer
MODEL_ID = Qwen/Qwen3.5-9B
TARGET_SOURCE_SET_IDS = ['hp_relation_chain_bridge_set_01']
required inputs checked = 6


## 2. 读取 working source set、taxonomy 和 payload


In [3]:
def read_csv(path):
    with path.open(newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def load_json(path):
    return json.loads(path.read_text(encoding='utf-8'))


source_set_rows = read_csv(SOURCE_SETS_PATH)
base_taxonomy_rows = read_csv(BASE_TAXONOMY_PATH)
batch2_taxonomy_rows = read_csv(BATCH2_ANNOTATION_PATH)

sampled_rows = load_json(SAMPLED_JSON_PATH)
expanded_rows = load_json(EXPANDED_JSON_PATH)
batch2_rows = load_json(BATCH2_FULL_JSON_PATH)

def parse_members(cell):
    return [x.strip() for x in str(cell).split('|') if x.strip()]


all_payload_rows = sampled_rows + expanded_rows + batch2_rows
task_payload_map = {}
for row in all_payload_rows:
    raw = row.get('raw') or row.get('full_example') or row
    task_payload_map[row['task_id']] = {'raw': raw}

taxonomy_map = {}
for row in base_taxonomy_rows:
    taxonomy_map[row['task_id']] = row

for row in batch2_taxonomy_rows:
    taxonomy_map[row['task_id']] = {
        'task_id': row['task_id'],
        'dataset': 'HotpotQA',
        'question': row['question'],
        'answer': row['answer'],
        'reasoning_label': row['reasoning_label'],
        'keep_drop': row['keep_drop'],
        'note': row['note'],
    }

requested_source_set_ids = list(TARGET_SOURCE_SET_IDS)
available_source_set_ids = {row['source_set_id'] for row in source_set_rows}
missing_source_set_ids = [source_set_id for source_set_id in requested_source_set_ids if source_set_id not in available_source_set_ids]
if missing_source_set_ids:
    raise FileNotFoundError(
        'Requested source set ids are missing from working source_sets.csv: ' + ', '.join(missing_source_set_ids)
    )

selected_source_sets = [row for row in source_set_rows if row['source_set_id'] in TARGET_SOURCE_SET_IDS]
if not selected_source_sets:
    raise ValueError('No source sets were selected. Check TARGET_SOURCE_SET_IDS and source_sets.csv.')

selected_member_ids = sorted({task_id for row in selected_source_sets for task_id in parse_members(row['member_task_ids'])})
missing_payload_ids = [task_id for task_id in selected_member_ids if task_id not in task_payload_map]
missing_taxonomy_ids = [task_id for task_id in selected_member_ids if task_id not in taxonomy_map]
if missing_payload_ids or missing_taxonomy_ids:
    diagnostic_lines = []
    if missing_payload_ids:
        diagnostic_lines.append('- missing payload rows: ' + ', '.join(missing_payload_ids))
    if missing_taxonomy_ids:
        diagnostic_lines.append('- missing taxonomy rows: ' + ', '.join(missing_taxonomy_ids))
    raise FileNotFoundError(
        'Selected source set members are not fully backed by loaded inputs:\n' + '\n'.join(diagnostic_lines)
    )

print('source set rows:', len(source_set_rows))
print('selected source sets:', [row['source_set_id'] for row in selected_source_sets])
print('payload rows available:', len(task_payload_map))
print('selected member ids:', selected_member_ids)


source set rows: 3
selected source sets: ['hp_relation_chain_bridge_set_01']
payload rows available: 52
selected member ids: ['hp_dev_1892', 'hp_dev_2485', 'hp_dev_5315', 'hp_dev_7220', 'hp_dev_7398']


## 3. 构造 source-set payload


In [4]:
def parse_members(cell):
    return [x.strip() for x in str(cell).split('|') if x.strip()]


def extract_support_sentences(raw):
    support = raw.get('supporting_facts', {})
    context = raw.get('context', {})
    titles = context.get('title', []) or []
    sentences = context.get('sentences', []) or []

    extracted = []
    for title, sent_id in zip(support.get('title', []), support.get('sent_id', [])):
        sentence_text = ''
        if title in titles:
            idx = titles.index(title)
            title_sents = sentences[idx]
            if 0 <= sent_id < len(title_sents):
                sentence_text = str(title_sents[sent_id]).strip()
        extracted.append({
            'title': title,
            'sent_id': sent_id,
            'sentence': sentence_text,
        })
    return extracted


def task_card(task_id):
    taxonomy = taxonomy_map[task_id]
    raw = task_payload_map[task_id]['raw']
    support_entries = extract_support_sentences(raw)
    support_titles = [entry['title'] for entry in support_entries]
    return {
        'task_id': task_id,
        'dataset': taxonomy.get('dataset', 'HotpotQA'),
        'reasoning_label': taxonomy['reasoning_label'],
        'question': taxonomy['question'],
        'answer': taxonomy['answer'],
        'taxonomy_note': taxonomy['note'],
        'raw_type': raw.get('type', ''),
        'level': raw.get('level', ''),
        'support_titles': support_titles,
        'support_entries': support_entries,
    }


def render_task_card(card):
    lines = []
    lines.append(f"- Task ID: {card['task_id']}")
    lines.append(f"  - Question: {card['question']}")
    lines.append(f"  - Answer: {card['answer']}")
    lines.append(f"  - Reasoning Label: {card['reasoning_label']}")
    lines.append(f"  - Raw Type: {card['raw_type']}")
    lines.append(f"  - Difficulty: {card['level']}")
    lines.append(f"  - Taxonomy Note: {card['taxonomy_note']}")
    if card['support_titles']:
        lines.append(f"  - Supporting Titles: {', '.join(card['support_titles'])}")
    nonempty_support = [entry for entry in card['support_entries'] if entry['sentence']]
    if nonempty_support:
        lines.append('  - Supporting Sentences:')
        for entry in nonempty_support:
            lines.append(f"    - [{entry['title']} / sent {entry['sent_id']}] {entry['sentence']}")
    return '\n'.join(lines)


def build_source_set_payload(source_set_row):
    member_ids = parse_members(source_set_row['member_task_ids'])
    cards = [task_card(task_id) for task_id in member_ids]
    rendered_cards = '\n\n'.join(render_task_card(card) for card in cards)
    return {
        'source_set_id': source_set_row['source_set_id'],
        'cluster': source_set_row['cluster'],
        'note': source_set_row['note'],
        'member_ids': member_ids,
        'cards': cards,
        'rendered_cards': rendered_cards,
    }


source_set_payloads = [build_source_set_payload(row) for row in selected_source_sets]

for payload in source_set_payloads:
    print(payload['source_set_id'], '->', payload['member_ids'])


hp_relation_chain_bridge_set_01 -> ['hp_dev_7398', 'hp_dev_2485', 'hp_dev_1892', 'hp_dev_5315', 'hp_dev_7220']


## 4. Prompt 模板


In [5]:
EPISODIC_SYSTEM_PROMPT = '''You are writing a reusable memory artifact for an LLM agent experiment.

Your job is to compress a source set into an episodic trace that still preserves episode-level specificity.

Hard constraints:
- Output markdown only.
- Preserve all five episodes as separate units.
- Do not turn this into a high-level principle list.
- Do not invent evidence that is not present.
- Do not mention this prompt or the experiment instructions.

The artifact should remain close to solved episodes while being shorter and cleaner than raw trajectories.'''

EPISODIC_USER_TEMPLATE = '''Write `episodic_trace.md` for the following frozen source set.

Source Set ID: {source_set_id}
Cluster: {cluster}
Source Set Note: {source_set_note}

Input episodes:
{rendered_cards}

Required output structure:

# Episodic Trace

## Source Set
- source_set_id: ...
- cluster: ...

## Episode Summaries
### 1. {first_task_id}
- question:
- answer:
- key lookup path:
- minimal support:
- reusable cue:

(repeat for all episodes)

## Local Pattern Notes
- 3 to 5 bullets only

Important distinction:
- keep this artifact episode-grounded
- preserve specific lookup paths and local cues
- do not collapse the whole set into generic advice
'''

CONSOLIDATION_SYSTEM_PROMPT = '''You are writing a reusable memory artifact for an LLM agent experiment.

Your job is to consolidate a source set into a cross-episode memory that captures shared structure, applicability, and boundaries.

Hard constraints:
- Output markdown only.
- Do not repeat all episodes one by one in detail.
- Synthesize across the set.
- Do not invent evidence that is not present.
- Do not mention this prompt or the experiment instructions.

The artifact should read like a compact reusable principle note, not an episode list.'''

CONSOLIDATION_USER_TEMPLATE = '''Write `cross_episode_consolidation.md` for the following frozen source set.

Source Set ID: {source_set_id}
Cluster: {cluster}
Source Set Note: {source_set_note}

Input episodes:
{rendered_cards}

Required output structure:

# Cross-Episode Consolidation

## Source Set
- source_set_id: ...
- cluster: ...

## Shared Structure
- 3 to 5 bullets

## Applicability
- when this memory is likely useful
- when it is not the right memory to use

## Operational Heuristic
- a short ordered checklist for applying this memory form

## Boundary / Failure Risk
- 2 to 4 bullets

Important distinction:
- this artifact must be meaningfully more abstract than `episodic_trace`
- it should emphasize shared structure, applicability, and boundary conditions
- avoid empty advice like "read carefully" or "reason step by step" unless grounded in the episodes above
'''


def build_episodic_user_prompt(payload):
    return EPISODIC_USER_TEMPLATE.format(
        source_set_id=payload['source_set_id'],
        cluster=payload['cluster'],
        source_set_note=payload['note'],
        rendered_cards=payload['rendered_cards'],
        first_task_id=payload['member_ids'][0],
    )


def build_consolidation_user_prompt(payload):
    return CONSOLIDATION_USER_TEMPLATE.format(
        source_set_id=payload['source_set_id'],
        cluster=payload['cluster'],
        source_set_note=payload['note'],
        rendered_cards=payload['rendered_cards'],
    )


## 5. 写 prompt preview


In [6]:
for payload in source_set_payloads:
    out_dir = ARTIFACTS_DIR / payload['source_set_id']
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / 'prompt_episodic_trace.md').write_text(build_episodic_user_prompt(payload), encoding='utf-8')
    (out_dir / 'prompt_cross_episode_consolidation.md').write_text(build_consolidation_user_prompt(payload), encoding='utf-8')

print('Prompt previews written.')


Prompt previews written.


## 6. 加载模型并生成 artifact


In [7]:
def infer_torch_dtype():
    if torch is None:
        raise ImportError('torch is not available.')
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16
    return torch.float32


def load_local_model(model_id):
    if torch is None or AutoTokenizer is None or AutoModelForCausalLM is None:
        raise ImportError('Required local generation libraries are not available.')

    token = HF_TOKEN or None
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {'device_map': 'auto', 'token': token}
    torch_dtype = infer_torch_dtype()

    if USE_4BIT:
        if BitsAndBytesConfig is None:
            raise ImportError('BitsAndBytesConfig is not available.')
        compute_dtype = torch_dtype if torch_dtype != torch.float32 else torch.float16
        model_kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=compute_dtype,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,
        )
    else:
        model_kwargs['torch_dtype'] = torch_dtype

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.eval()
    return tokenizer, model, torch_dtype


def build_chat_text(system_prompt, user_prompt):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=ENABLE_THINKING)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def get_model_device(model_obj):
    return next(model_obj.parameters()).device


def generate_markdown(system_prompt, user_prompt):
    prompt_text = build_chat_text(system_prompt, user_prompt)
    model_inputs = tokenizer([prompt_text], return_tensors='pt')
    model_device = get_model_device(model)
    model_inputs = {k: v.to(model_device) for k, v in model_inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=DO_SAMPLE,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    output_ids = generated_ids[0][model_inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    if not ENABLE_THINKING:
        text = text.replace('<think>', '').replace('</think>', '').strip()
    return text


tokenizer = None
model = None
torch_dtype = None
if RUN_GENERATION:
    tokenizer, model, torch_dtype = load_local_model(MODEL_ID)
    print('Loaded model =', MODEL_ID)
    print('Loaded dtype =', torch_dtype)
else:
    print('RUN_GENERATION = False -> prompt preview only')

generation_manifest = []
for payload in source_set_payloads:
    out_dir = ARTIFACTS_DIR / payload['source_set_id']
    episodic_user_prompt = build_episodic_user_prompt(payload)
    consolidation_user_prompt = build_consolidation_user_prompt(payload)

    if not RUN_GENERATION:
        generation_manifest.append({'source_set_id': payload['source_set_id'], 'artifact_type': 'episodic_trace', 'status': 'prompt_only', 'model': MODEL_ID, 'backend': 'huggingface_transformers', 'output_file': str(out_dir / 'episodic_trace.md'), 'char_count': '', 'error': ''})
        generation_manifest.append({'source_set_id': payload['source_set_id'], 'artifact_type': 'cross_episode_consolidation', 'status': 'prompt_only', 'model': MODEL_ID, 'backend': 'huggingface_transformers', 'output_file': str(out_dir / 'cross_episode_consolidation.md'), 'char_count': '', 'error': ''})
        continue

    for artifact_type, system_prompt, user_prompt, filename in [
        ('episodic_trace', EPISODIC_SYSTEM_PROMPT, episodic_user_prompt, 'episodic_trace.md'),
        ('cross_episode_consolidation', CONSOLIDATION_SYSTEM_PROMPT, consolidation_user_prompt, 'cross_episode_consolidation.md'),
    ]:
        try:
            text = generate_markdown(system_prompt, user_prompt)
            (out_dir / filename).write_text(text + '\n', encoding='utf-8')
            generation_manifest.append({'source_set_id': payload['source_set_id'], 'artifact_type': artifact_type, 'status': 'generated', 'model': MODEL_ID, 'backend': 'huggingface_transformers', 'output_file': str(out_dir / filename), 'char_count': len(text), 'error': ''})
        except Exception as e:
            generation_manifest.append({'source_set_id': payload['source_set_id'], 'artifact_type': artifact_type, 'status': 'error', 'model': MODEL_ID, 'backend': 'huggingface_transformers', 'output_file': str(out_dir / filename), 'char_count': '', 'error': str(e)[:300]})
            print('Generation error:', payload['source_set_id'], artifact_type, e)

with MANIFEST_PATH.open('w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['source_set_id', 'artifact_type', 'status', 'model', 'backend', 'output_file', 'char_count', 'error'])
    writer.writeheader()
    for row in generation_manifest:
        writer.writerow(row)

print('Wrote manifest to:', MANIFEST_PATH)
for row in generation_manifest:
    print(row)

if model is not None:
    gc.collect()
    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()


`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Loaded model = Qwen/Qwen3.5-9B
Loaded dtype = torch.bfloat16
Wrote manifest to: /root/2026_SelectTransfer/artifacts/relation_chain_artifact_generation_manifest.csv
{'source_set_id': 'hp_relation_chain_bridge_set_01', 'artifact_type': 'episodic_trace', 'status': 'generated', 'model': 'Qwen/Qwen3.5-9B', 'backend': 'huggingface_transformers', 'output_file': '/root/2026_SelectTransfer/artifacts/hp_relation_chain_bridge_set_01/episodic_trace.md', 'char_count': 3467, 'error': ''}
{'source_set_id': 'hp_relation_chain_bridge_set_01', 'artifact_type': 'cross_episode_consolidation', 'status': 'generated', 'model': 'Qwen/Qwen3.5-9B', 'backend': 'huggingface_transformers', 'output_file': '/root/2026_SelectTransfer/artifacts/hp_relation_chain_bridge_set_01/cross_episode_consolidation.md', 'char_count': 3369, 'error': ''}


## 7. 预览生成结果


In [8]:
for payload in source_set_payloads:
    out_dir = ARTIFACTS_DIR / payload['source_set_id']
    print('=' * 80)
    print(payload['source_set_id'])
    for filename in ['episodic_trace.md', 'cross_episode_consolidation.md']:
        path = out_dir / filename
        print('-' * 40)
        print(filename, 'exists =', path.exists())
        if path.exists():
            text = path.read_text(encoding='utf-8')
            print(text[:1200])
            print()


hp_relation_chain_bridge_set_01
----------------------------------------
episodic_trace.md exists = True
# Episodic Trace

## Source Set
- source_set_id: hp_relation_chain_bridge_set_01
- cluster: bridge

## Episode Summaries
### 1. hp_dev_7398
- question: Who was the brother of the wife of the Democratic Party nomination for Vice President in 1972?
- answer: President John F. Kennedy
- key lookup path: 1972 VP Nominee (Sargent Shriver) -> Wife (Eunice Kennedy Shriver) -> Brother (John F. Kennedy)
- minimal support: Sargent Shriver was the nominee; his wife was Eunice Kennedy Shriver; her brother was President John F. Kennedy.
- reusable cue: Kinship chains often require traversing a spouse link to reach the target relative.

### 2. hp_dev_2485
- question: Who is the mother of Mary, Crown Princess of Denmark's husband?
- answer: Queen Margrethe II
- key lookup path: Mary (Crown Princess) -> Husband (Frederik, Crown Prince) -> Mother (Queen Margrethe II)
- minimal support: Mary is the w